_ExplainDB — Copyright (C) 2026 Prof. Dr. Jens Dittrich, Saarland University._  
_Licensed under the GNU Affero General Public License v3 (AGPL-3.0). See the `LICENSE` file._

In [1]:
import sys, pathlib

# Put the repo root (the directory containing `system/`) on sys.path so
# `from system...` works whether this notebook is opened from its own
# folder or from the repo root.
for _p in (pathlib.Path.cwd(), *pathlib.Path.cwd().parents):
    if (_p / "system").is_dir():
        if str(_p) not in sys.path:
            sys.path.insert(0, str(_p))
        break

**Data layouts: row vs. column**

This notebook shows the effects of switching from row to column layout. For educational purposes we use plain text files, written as CSV (comma-separated values) with one value per line, so the files contain no commas. Note that in a real database, we would use a binary format. Note that these effects do not only happen when writing data to files (**serializing**) but at any time when we **linearize** data, i.e. every time we decide how to map higher-dimensional data to a one-dimensional data space.

For two queries, we count exactly how much of each file the query has to read. The animation [row-or-column](https://htmlpreview.github.io/?https://github.com/explaindb/explaindb/blob/main/visualizations/row-or-column.html) shows the same effects in main memory.

**Create some fake data**

In [2]:
from faker import Faker

from system.data_classes import Book

Faker.seed(42)
fake: Faker = Faker()

number_of_tuples: int = 10000

# create some fake data mimicking tuples in a relation
books: list[Book] = [
    Book(
        fake.sentence(),
        fake.name(),
        fake.pyfloat(min_value=0, max_value=100, right_digits=2),
    )
    for _ in range(number_of_tuples)
]
books[:10]

[Book(title='Agent every development say.', author='Stephanie Miller', price=54.4),
 Book(title='Behavior discussion own.', author='Michelle Miles', price=53.28),
 Book(title='Operation speak according south recently future choice.', author='Gabrielle Davis', price=12.45),
 Book(title='Page southern role movie win her.', author='Sandra Montgomery', price=46.73),
 Book(title='Relate animal direction eye bag.', author='Brian Ramirez', price=81.46),
 Book(title='Have decide environment view.', author='Lisa Jackson', price=93.31),
 Book(title='Left establish understand read.', author='Frederick Tate', price=7.29),
 Book(title='Source husband at.', author='Latoya Robbins', price=27.83),
 Book(title='How trip learn enter east no enjoy.', author='Stephanie Ross', price=28.17),
 Book(title='Management sense technology check civil quite others.', author='Kyle Mcdonald', price=32.7)]

In [3]:
from pandas import DataFrame

# store data in pandas dataframe:
schema: list[str] = ["title", "author", "price"]
df: DataFrame = DataFrame(data=books, columns=schema)
df

,title,author,price
0,Agent every development say.,Stephanie Miller,54.40
1,Behavior discussion own.,Michelle Miles,53.28
2,Operation speak according south recently futur...,Gabrielle Davis,12.45
3,Page southern role movie win her.,Sandra Montgomery,46.73
4,Relate animal direction eye bag.,Brian Ramirez,81.46
...,...,...,...
9995,Remember result act.,Mark Clark,29.10
9996,Tonight list Democrat year forward.,Danny Walker,57.94
9997,President star room partner.,Jennifer Chavez,0.26
9998,Well bag develop support.,Katherine Ramos,6.50


**Emulate row store**

To emulate a row-major layout, we write the data into a CSV file where each tuple is stored consecutively. Within each tuple, each attribute is also stored consecutively. Inspect the generated `row.csv` file for more detail.

Each value takes one line, so the value of attribute `a` of book `t` is on line `3·t + a` of `row.csv` (books, attributes and lines all counted from 0; note that `head` and most editors count lines from 1).

In [4]:
# write data into file in row-major layout to emulate store; each value takes one line, ended by "\n" on every
# system (on Windows, pandas would otherwise end each tuple with "\r\n"):
path_row: str = "row.csv"
df.to_csv(
    path_or_buf=path_row,
    sep="\n",
    lineterminator="\n",
    header=False,
    index=False,
    mode="w",
)

In [5]:
!head row.csv

Agent every development say.
Stephanie Miller
54.4
Behavior discussion own.
Michelle Miles
53.28
Operation speak according south recently future choice.
Gabrielle Davis
12.45
Page southern role movie win her.


**Emulate column store**

To emulate a column-major layout, we write the data into a CSV file where each attribute column is stored consecutively. Inspect the generated `column.csv` file for more detail.

Here, the value of attribute `a` of book `t` is on line `10,000·a + t` of `column.csv`.

In [6]:
# write data into file in column-major layout to emulate store; each value takes one line, ended by "\n" on
# every system:
path_column: str = "column.csv"
attribute_index: int
col: str
for attribute_index, col in enumerate(df.columns):
    df.to_csv(
        path_or_buf=path_column,
        columns=[col],
        lineterminator="\n",
        header=False,
        index=False,
        mode=(
            "w" if attribute_index == 0 else "a"
        ),  # truncate file before first column, append afterward
    )

In [7]:
!head column.csv

Agent every development say.
Behavior discussion own.
Operation speak according south recently future choice.
Page southern role movie win her.
Relate animal direction eye bag.
Have decide environment view.
Left establish understand read.
Source husband at.
How trip learn enter east no enjoy.
Management sense technology check civil quite others.


**How much a query has to read**

Disks, SSDs (solid-state drives) and the operating system do not read single bytes of a file but whole blocks, here of 4,096 bytes. A query therefore has to read every block that holds at least one byte of a value it needs. Main memory, too, is read in blocks, the cache lines, typically of 64 bytes, so the same effects occur there (see the animation linked above, which uses fewer and shorter books, so its numbers differ).

`ValueFile` reads a file this way: it reads each such block once, takes the values the query needs from these blocks, and counts the blocks and the places in the file they lie in (blocks that follow each other are one place). A hard disk needs a seek of a few milliseconds to reach each place (if the file lies in one piece on the disk) and then reads its blocks one after the other. `ValueFile` knows at which byte each value starts and ends, as a database does (it stores values of a fixed width or keeps the position of each value); for our CSV files, it finds these positions once, when it is made, and does not count that read. A real database whose values differ in length also has to read the positions it needs, which can cost further blocks (e.g. when it keeps the positions in an array of their own). The line breaks only separate the values in the file, so a query does not need them.

We do not measure the time of the queries: the files were just written, so the operating system still holds them in main memory, and a timing would not measure the disk at all. Besides, a query on files of half a megabyte takes a few milliseconds, and such timings vary from run to run.

In [8]:
from system.playground.data_layout import (
    BLOCK_SIZE,
    BlockRead,
    ValueFile,
    column_major_line,
    row_major_line,
)

# the two files, each with the bytes at which each of its values starts and ends:
row_file: ValueFile = ValueFile(path_row)
column_file: ValueFile = ValueFile(path_column)
print("Block size:", BLOCK_SIZE, "bytes")
print(f"row.csv: {row_file.size:,} bytes in {row_file.number_of_blocks} blocks")
print(
    f"column.csv: {column_file.size:,} bytes in {column_file.number_of_blocks} blocks"
)

Block size: 4096 bytes
row.csv: 571,549 bytes in 140 blocks
column.csv: 571,549 bytes in 140 blocks


**Consider the following query accessing one whole tuple**
``` sql
SELECT *
FROM books
LIMIT 1
OFFSET 100
```

This query returns all three attributes of one book: the book at position 100, i.e. the 101st book stored (without `ORDER BY`, SQL (Structured Query Language) leaves open which book that is; here it is the order in which the books are stored). In the row layout, the three values of the book lie next to each other in `row.csv`. In the column layout, they lie in three places of `column.csv`, one in each column. So we expect the row layout to read fewer blocks.

In [9]:
# the position of the book the query returns:
book_index: int = 100

# execute the query given the row layout: the lines of the book's three values follow each other:
row_lines: list[int] = [
    row_major_line(book_index, attribute_index, number_of_attributes=len(schema))
    for attribute_index in range(len(schema))
]
row_read: BlockRead = row_file.read(row_lines)
print("Book:", row_read.values)
print("Row layout:", row_read.summary())

Book: ('War analysis situation term miss.', 'Victor Brown', '61.51')
Row layout: 3 values (50 B) needed, 1 block read (4,096 B) from 1 place


In [10]:
# execute the query given the column layout: the lines of the book's three values lie in three columns:
column_lines: list[int] = [
    column_major_line(book_index, attribute_index, number_of_tuples=number_of_tuples)
    for attribute_index in range(len(schema))
]
column_read: BlockRead = column_file.read(column_lines)
print("Book:", column_read.values)
print("Column layout:", column_read.summary())

Book: ('War analysis situation term miss.', 'Victor Brown', '61.51')
Column layout: 3 values (50 B) needed, 3 blocks read (12,288 B) from 3 places


Both layouts return the same book and need the same 50 bytes, but the row layout reads 1 block and the column layout 3 blocks from 3 places, one for each attribute. Each further attribute costs the column layout about one more block, and on a hard disk one more seek; this matters when a query needs only a few tuples. The animation shows the same effect in main memory: one whole book takes 1 cache line of the row layout but 3 of the column layout.

Needing several attributes of the same tuples is not always enough for the row layout to read less, though. `SELECT author, price FROM books LIMIT 314 OFFSET 100` needs two of the three attributes of 314 books: the row layout reads 5 blocks from 1 place, the column layout only 4 blocks, but from 2 places (authors and prices). Every block of `row.csv` also holds titles, the longest values, which this query does not need. Which layout is faster for this query depends on the device: on a hard disk, the second seek of the column layout costs more time than reading the block it saves; on an SSD or in main memory, reaching another place costs little, so the 4 blocks of the column layout are about as fast or faster. Try it, e.g. for the row layout:
``` python
row_file.read(
    row_major_line(t, a, number_of_attributes=len(schema))
    for t in range(100, 414)
    for a in (1, 2)
).summary()
```

**Consider the following query accessing and aggregating a single attribute**
``` sql
SELECT MAX(price)
FROM books
```

This query needs a single attribute (price), but of all 10,000 books. In the row layout, the price of each book lies between its author and the title of the next book, so every block of `row.csv` holds prices and the query has to read the whole file. In the column layout, all prices lie next to each other at the end of `column.csv`, so the query reads only the blocks of the price column. Therefore, we now expect the column layout to read far fewer blocks.

In [11]:
# the position of the attribute price in the schema:
price_index: int = schema.index("price")

# execute the query given the row layout: every third line holds a price:
row_prices_read: BlockRead = row_file.read(
    row_major_line(t, price_index, number_of_attributes=len(schema))
    for t in range(number_of_tuples)
)
# compute the aggregate over the prices, read as numbers:
aggregate: float = max(float(price) for price in row_prices_read.values)
print("Aggregate:", aggregate)
print("Row layout:", row_prices_read.summary())

Aggregate: 99.98
Row layout: 10,000 values (47,014 B) needed, 140 blocks read (571,549 B) from 1 place


In [12]:
# execute the query given the column layout: the prices are the last 10,000 lines:
column_prices_read: BlockRead = column_file.read(
    column_major_line(t, price_index, number_of_tuples=number_of_tuples)
    for t in range(number_of_tuples)
)
# compute the aggregate over the prices, read as numbers:
aggregate = max(float(price) for price in column_prices_read.values)
print("Aggregate:", aggregate)
print("Column layout:", column_prices_read.summary())

Aggregate: 99.98
Column layout: 10,000 values (47,014 B) needed, 15 blocks read (59,549 B) from 1 place


Both layouts need the same 47,014 bytes, the 10,000 prices, and read them from 1 place, but the row layout reads 140 blocks, the whole file, and the column layout only 15 (not 14, as the price column starts in the middle of a block). A book takes about ten times as many bytes in the file as its price, so the row layout reads about ten times as much. The animation shows the same effect in main memory: there a book takes 32 bytes and its price 8, so `SELECT MAX(price)` loads 4 cache lines of the row layout but only 1 of the column layout. Note that the column layout has further advantages for such queries: the values of a column all have the same type, so they compress well, and we could additionally speed up the execution by utilizing vectorization and SIMD (single instruction, multiple data) instructions, which process several values of a column at once.

**Tests**

The tests check that both layouts give the same answers as the data, and every number quoted above about the two files.

In [13]:
import math
import unittest

from system.playground.notebook_tests import run_notebook_tests


class DataLayoutTest(unittest.TestCase):
    """The answers of both layouts and the numbers the text quotes."""

    def test_both_layouts_return_the_book(self) -> None:
        """SELECT * ... LIMIT 1 OFFSET 100 returns title, author and price of book 100 in both layouts."""
        book: Book = books[book_index]
        for read in (row_file.read(row_lines), column_file.read(column_lines)):
            title, author, price = read.values
            self.assertEqual(
                (book.title, book.author, book.price), (title, author, float(price))
            )

    def test_both_layouts_return_the_maximum(self) -> None:
        """SELECT MAX(price) returns the highest price of all books in both layouts."""
        self.assertEqual(df["price"].max(), aggregate)
        for read in (row_prices_read, column_prices_read):
            self.assertEqual(aggregate, max(float(price) for price in read.values))

    def test_numbers_of_one_whole_book(self) -> None:
        """One whole book: the same 50 bytes, 1 block of the row layout, 3 blocks from 3 places of the column
        layout; both files have the same size."""
        self.assertEqual(row_file.size, column_file.size)
        row_book: BlockRead = row_file.read(row_lines)
        column_book: BlockRead = column_file.read(column_lines)
        self.assertEqual((50, 50), (row_book.bytes_needed, column_book.bytes_needed))
        self.assertEqual((1, 1), (row_book.blocks_read, row_book.places))
        self.assertEqual((3, 3), (column_book.blocks_read, column_book.places))

    def test_numbers_of_authors_and_prices(self) -> None:
        """Author and price of 314 books: the same values, 5 blocks from 1 place of the row layout, 4 blocks from 2
        places of the column layout."""
        row_authors_prices: BlockRead = row_file.read(
            row_major_line(t, a, number_of_attributes=len(schema))
            for t in range(100, 414)
            for a in (1, 2)
        )
        column_authors_prices: BlockRead = column_file.read(
            column_major_line(t, a, number_of_tuples=number_of_tuples)
            for t in range(100, 414)
            for a in (1, 2)
        )
        # both layouts return author and price of books 100 to 413, in the same order:
        expected: list[str] = [
            value for book in books[100:414] for value in (book.author, str(book.price))
        ]
        self.assertEqual(expected, list(row_authors_prices.values))
        self.assertEqual(expected, list(column_authors_prices.values))
        # the blocks and places of each layout:
        self.assertEqual(
            (5, 1), (row_authors_prices.blocks_read, row_authors_prices.places)
        )
        self.assertEqual(
            (4, 2), (column_authors_prices.blocks_read, column_authors_prices.places)
        )

    def test_numbers_of_all_prices(self) -> None:
        """All prices: the same 47,014 bytes from 1 place, the whole row file (140 blocks), 15 blocks of the
        column file, 14 if the price column started at a block; about ten times as much read in the row layout.
        """
        self.assertEqual(
            (47_014, 47_014),
            (row_prices_read.bytes_needed, column_prices_read.bytes_needed),
        )
        self.assertEqual((1, 1), (row_prices_read.places, column_prices_read.places))
        self.assertEqual(
            (140, 15), (row_prices_read.blocks_read, column_prices_read.blocks_read)
        )
        self.assertEqual(
            (row_file.number_of_blocks, row_file.size),
            (row_prices_read.blocks_read, row_prices_read.bytes_read),
        )
        # the bytes from the first to the last price of the column file fit into 14 blocks:
        first_price: int = column_major_line(
            0, price_index, number_of_tuples=number_of_tuples
        )
        price_bytes: int = column_file.ends[-1] - column_file.starts[first_price]
        self.assertEqual(14, math.ceil(price_bytes / BLOCK_SIZE))
        # a book takes about ten times as many bytes as its price, each with its line break, and so the row layout
        # reads about ten times as much:
        self.assertEqual(
            10,
            round(
                column_file.size / (column_file.size - column_file.starts[first_price])
            ),
        )
        self.assertEqual(
            10, round(row_prices_read.bytes_read / column_prices_read.bytes_read)
        )


run_notebook_tests(DataLayoutTest)

test_both_layouts_return_the_book (__main__.DataLayoutTest.test_both_layouts_return_the_book)
SELECT * ... LIMIT 1 OFFSET 100 returns title, author and price of book 100 in both layouts. ... 

ok


test_both_layouts_return_the_maximum (__main__.DataLayoutTest.test_both_layouts_return_the_maximum)
SELECT MAX(price) returns the highest price of all books in both layouts. ... 

ok


test_numbers_of_all_prices (__main__.DataLayoutTest.test_numbers_of_all_prices)
All prices: the same 47,014 bytes from 1 place, the whole row file (140 blocks), 15 blocks of the ... 

ok


test_numbers_of_authors_and_prices (__main__.DataLayoutTest.test_numbers_of_authors_and_prices)
Author and price of 314 books: the same values, 5 blocks from 1 place of the row layout, 4 blocks from 2 ... 

ok


test_numbers_of_one_whole_book (__main__.DataLayoutTest.test_numbers_of_one_whole_book)
One whole book: the same 50 bytes, 1 block of the row layout, 3 blocks from 3 places of the column ... 

ok


----------------------------------------------------------------------
Ran 5 tests in 0.006s

OK


In [14]:
# remove the two files written above; to run further queries, run the cells that write them again first:
!rm row.csv column.csv